# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library and the dataset's Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access the Croissant metadata
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', None)}\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview
List available record sets in the dataset and their fields using `@id`. This helps decide what data to extract later.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"\nRecord sets found in dataset (by @id):\n---")
if not record_sets:
    print("(No record sets were found directly; this may indicate the schema nests records differently or as a single set. Attempting to infer main tables from columns...)")
    # Try to infer record set from dataset.fields
    fields = list(dataset.fields)
    if fields:
        print("Found the following fields in the dataset (by @id):")
        for f in fields:
            print(f"- {getattr(f, '@id', None)} (name={getattr(f, 'name', None)}, dataType={getattr(f, 'data_type', None)})")
    else:
        print("No record sets or fields found in schema. Please review the Croissant definition.")
else:
    for record_set in record_sets:
        print(f"@id: {getattr(record_set, '@id', None)} | name: {getattr(record_set, 'name', None)}")
        print("  Fields:")
        for field in getattr(record_set, 'fields', []):
            print(f"    - @id: {getattr(field, '@id', None)} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from the main record set or from fields (as inferred), referencing each entity by its `@id`.

If the dataset has a main record set, we'll use its `@id` to extract records.

In [ ]:
# Identify the main table. 
# The Croissant schema does not explicitly list record sets, so we try to extract all records.

# Try standard record set IDs (replace if changed; see overview section for what IDs are found)
record_sets = list(dataset.record_sets)
dataframes = {}

if record_sets:
    record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
else:
    # If not present, Croissant schema may use fields directly in one root table
    record_set_ids = [None]

for record_set_id in record_set_ids:
    try:
        records_generator = dataset.records(record_set=record_set_id) if record_set_id is not None else dataset.records()
        records = list(records_generator)
        # If there are records, store as DataFrame
        if records:
            key = record_set_id if record_set_id is not None else 'main'
            dataframes[key] = pd.DataFrame(records)
    except Exception as e:
        print(f"Failed to extract records for record_set_id={record_set_id}: {e}")

if not dataframes:
    print("No records could be loaded using mlcroissant. Check dataset definition.")
else:
    for k, df in dataframes.items():
        print(f"\nRecord set: {k}\nColumns: {df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Process and analyze the data. We will:
- Select a numeric field (e.g., `MW [kDa]`, `Coverage [\%]`, `Unique Peptides` if present).
- Filter, normalize, and group the data.

The field `@id` will be referenced explicitly.

In [ ]:
# Choose the first (only) table for EDA
import numpy as np
main_key = list(dataframes.keys())[0]
df = dataframes[main_key]

# For the FAIR² dataset, likely columns are: 'Accession', 'Description', 'Coverage [%]', 'Unique Peptides', 'MW [kDa]', 'pI', etc.
# Let's pick 'MW [kDa]' as a numeric field for demonstration. Use the actual column name if different.

# Find a numerical column as example
numeric_field_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]]
if not numeric_field_candidates:
    # Try to coerce typical columns to numeric
    candidates = ['MW [kDa]', 'Coverage [%]', 'Unique Peptides', 'Total Peptides', 'pI']
    for c in candidates:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            if df[c].notnull().sum() > 0:
                numeric_field_candidates.append(c)

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0] # e.g., 'MW [kDa]'
    print(f"Using numeric field: {numeric_field}")
else:
    raise ValueError("No suitable numeric field found in the main dataframe.")

# Select a threshold for demonstration
threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (using mean as threshold):")
print(filtered_df.head())

# Normalize the numeric field
col_norm = numeric_field + '_normalized'
filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, col_norm]].head())

# Choose a group field, e.g. 'Description' if categorical, or other suitable
group_field_candidates = [col for col in df.columns if col.lower().startswith('desc') or col.lower() == 'accession']
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nMean of {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Plot numeric field distributions and show relationships with another variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there's another numeric, do pairplot
if len(numeric_field_candidates) >= 2:
    other_numeric = numeric_field_candidates[1]
    plt.figure(figsize=(6,6))
    sns.scatterplot(data=df, x=numeric_field, y=other_numeric)
    plt.title(f"{numeric_field} vs {other_numeric}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore a Croissant-annotated dataset using the `mlcroissant` library, referencing all entities by their `@id` where possible. Key steps included:
- Loading metadata and records
- Inspecting record set and field structure
- Extracting data and processing numeric fields
- Visualizing data distributions

This workflow supports FAIR and reproducible data science using mass spectrometry data, and can be adapted to any properly-tagged Croissant dataset.